# 🌲 Bagging — Bootstrap Aggregating
### *An End-to-End ML Learning Resource: Theory → Scratch → Sklearn → Interview Prep*

---

## 📌 Learning Objectives
By the end of this notebook you will be able to:
- Explain **why** bagging reduces variance without touching bias
- Derive the ensemble variance formula: $\text{Var}(\bar{f}) = \rho\sigma^2 + \frac{(1-\rho)\sigma^2}{T}$
- **Implement Bagging from scratch** using NumPy (class-based, with OOB tracking)
- **Implement Bagging with Scikit-Learn** and tune key hyperparameters
- Evaluate ensembles with accuracy, AUC-ROC, and OOB score
- Answer the **top interview questions** on bagging with depth and precision

---

## 🔑 Prerequisites
| Topic | Why It's Needed |
|---|---|
| Decision Trees (CART) | Default base learner for bagging |
| Bias-Variance Tradeoff | Core reason bagging exists |
| Bootstrap Sampling | The "B" in Bagging |
| Cross-Validation | Benchmark for OOB estimates |
| Numpy & Sklearn basics | Implementation tooling |

---

## 📂 Dataset
**Heart Disease UCI** — Real-world clinical dataset predicting the presence of heart disease.

🔗 **Kaggle Link:** https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset

**Why this dataset?**
- Binary classification (ideal for demonstrating majority-vote aggregation)
- Mix of numerical and categorical features (realistic preprocessing required)
- Well-known benchmark, enabling result comparison with published benchmarks
- 303 rows — small enough to clearly demonstrate OOB mechanics

---

## 📚 Credits & References
| Resource | Link |
|---|---|
| Breiman (1996) — Original Bagging Paper | https://link.springer.com/article/10.1007/BF00058655 |
| Scikit-Learn Ensemble Guide | https://scikit-learn.org/stable/modules/ensemble.html |
| ESL Chapter 8 & 15 (Hastie, Tibshirani, Friedman) | https://hastie.su.domains/ElemStatLearn/ |
| StatQuest: Random Forests | https://www.youtube.com/watch?v=J4Wdy0Wc_xQ |

---
*Author perspective: Senior ML Engineer (FAANG) + MIT Educator*


## 📦 Cell 2: Library Imports

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# ── Numerical Computing ────────────────────────────────────────────────────────
import numpy as np                          # Array ops, bootstrap sampling, random seed

# ── Data Manipulation ──────────────────────────────────────────────────────────
import pandas as pd                         # DataFrame loading, EDA, preprocessing

# ── Visualization ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt             # Low-level plot control
import seaborn as sns                       # Statistical visualizations, heatmaps

# ── Scikit-Learn: Preprocessing ───────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold

# ── Scikit-Learn: Models ───────────────────────────────────────────────────────
from sklearn.tree import DecisionTreeClassifier   # Base learner for bagging
from sklearn.ensemble import BaggingClassifier    # Sklearn Bagging implementation

# ── Scikit-Learn: Metrics ─────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score,          # Primary classification metric
    roc_auc_score,           # Discrimination ability (threshold-independent)
    classification_report,   # Precision, recall, F1 per class
    confusion_matrix,        # Visualise error breakdown
    roc_curve                # ROC curve coordinates for plotting
)

# ── Global Style ───────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
SEED = 42
np.random.seed(SEED)
print("✅ All libraries imported successfully. SEED =", SEED)


---
## 🧠 Part 1 — Theory Recap

### What Is Bagging?
**Bootstrap AGGregating (Bagging)** — introduced by Leo Breiman in 1996 — is an ensemble method that trains *T* base learners on *T* different bootstrap samples of the training data and combines their predictions via majority vote (classification) or averaging (regression).

### Core Intuition
> *Ask 100 doctors the same question, each seeing a different random 63% of the patient's file. Their individual errors are uncorrelated — so when you tally the votes, random mistakes cancel out.*

### 5 Key Concepts

1. **Bootstrap Sampling** — Draw *n* samples *with replacement* from a dataset of size *n*. Each bag contains ~63.2% unique rows; the rest are duplicates. The left-out ~36.8% form the **Out-of-Bag (OOB)** set.

2. **Variance Reduction via Averaging** — For *T* independent models each with variance $\sigma^2$, the averaged prediction has variance $\sigma^2 / T$. In practice models are correlated (share the same source data), so the formula becomes:
$$\text{Var}(\bar{f}) = \rho\sigma^2 + \frac{(1-\rho)\,\sigma^2}{T}$$
where $\rho$ is the average pairwise inter-model correlation. **The floor $\rho\sigma^2$ never disappears** — decorrelating models matters more than adding more trees once *T* is large.

3. **Bias Is Untouched** — Bagging averages predictions; it cannot remove systematic error. If the base learner is biased (underfits), the ensemble will also be biased. Bagging is a **variance reduction** tool, not a bias reduction tool.

4. **OOB Evaluation** — Each training sample is excluded from ~36.8% of trees. Predicting each sample using only its OOB trees yields an unbiased test-set estimate **without a separate validation split**.

5. **Parallelism** — Because each model is trained independently on its own bootstrap bag, bagging is **trivially parallelizable** (`n_jobs=-1` in sklearn). This contrasts sharply with boosting, which is inherently sequential.

### When Does Bagging Shine?
| Model Property | Bagging Benefit |
|---|---|
| High variance (deep tree, KNN, neural net) | Large variance reduction |
| Low bias (already fits training data well) | Bias stays near zero after averaging |
| Noisy labels | Noise partially cancels across bags |
| Sufficient data (n ≥ a few hundred) | Diverse bags, reliable OOB estimates |


---
## 📂 Part 1 — Load & Explore the Dataset
We use the **Heart Disease UCI** dataset (303 rows × 14 columns).  
**Target:** `target` — 1 = presence of heart disease, 0 = absence.  
Download `heart.csv` from https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset and place it in your working directory.


In [ ]:
# ── Load Dataset ─────────────────────────────────────────────────────────────
# The dataset ships from Kaggle as 'heart.csv'.
# If you're running locally, ensure heart.csv is in the same folder as this notebook.
df = pd.read_csv('heart.csv')

print("=" * 55)
print("DATASET SHAPE:", df.shape)
print("=" * 55)

# ── First Look ────────────────────────────────────────────────────────────────
print("\n🔍 First 5 rows:")
display(df.head())

print("\n📋 Data Types & Non-Null Counts:")
df.info()

print("\n📊 Descriptive Statistics:")
display(df.describe().round(2))

# ── Target Distribution ───────────────────────────────────────────────────────
print("\n🎯 Target Distribution:")
print(df['target'].value_counts().rename({1: 'Disease (1)', 0: 'No Disease (0)'}))
print(f"\nClass balance: {df['target'].mean():.2%} positive class")

# ── Feature Glossary ──────────────────────────────────────────────────────────
glossary = {
    'age': 'Age in years',
    'sex': '1 = Male, 0 = Female',
    'cp': 'Chest pain type (0–3)',
    'trestbps': 'Resting blood pressure (mm Hg)',
    'chol': 'Serum cholesterol (mg/dl)',
    'fbs': 'Fasting blood sugar > 120 mg/dl (1 = True)',
    'restecg': 'Resting ECG results (0–2)',
    'thalach': 'Maximum heart rate achieved',
    'exang': 'Exercise-induced angina (1 = Yes)',
    'oldpeak': 'ST depression induced by exercise',
    'slope': 'Slope of peak exercise ST segment',
    'ca': 'Number of major vessels colored by fluoroscopy (0–3)',
    'thal': '3=Normal, 6=Fixed defect, 7=Reversible defect',
    'target': '1 = Heart Disease, 0 = No Heart Disease'
}
print("\n📖 Feature Glossary:")
for col, desc in glossary.items():
    print(f"  {col:<12} → {desc}")


---
## ⚙️ Data Preprocessing
Steps:
1. **Check for missing values** — handle if present
2. **Encode categoricals** — `cp`, `thal`, `slope` are ordinal/nominal
3. **Scale numerics** — ensures distance-based base learners (if used) aren't dominated by large-range features
4. **Train / Test Split** — stratified 80/20 split to preserve class balance


In [ ]:
np.random.seed(SEED)

# ── 1. Missing Values ─────────────────────────────────────────────────────────
print("Missing values per column:")
print(df.isnull().sum())
# Heart Disease UCI is clean; no imputation needed — but always check!

# ── 2. Feature / Target Split ─────────────────────────────────────────────────
X = df.drop('target', axis=1)   # 13 features
y = df['target']                 # binary label

# ── 3. Scale Numerical Features ───────────────────────────────────────────────
# Bagging of decision trees does NOT require scaling (trees are scale-invariant),
# but we scale anyway so the notebook is reusable with other base learners (e.g. SVM, KNN).
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns
)

# ── 4. Train / Test Split ─────────────────────────────────────────────────────
# stratify=y ensures both splits have the same class ratio — critical for medical datasets.
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print(f"\n✅ Preprocessing Complete")
print(f"   Training set : {X_train.shape[0]} rows")
print(f"   Test set     : {X_test.shape[0]} rows")
print(f"   Features     : {X_train.shape[1]}")
print(f"   Train positive rate: {y_train.mean():.2%}")
print(f"   Test  positive rate: {y_test.mean():.2%}")


---
## 🔨 Part 2 — From-Scratch Implementation

### Architecture Design
We implement a `BaggingClassifierScratch` class with:

| Method | Responsibility |
|---|---|
| `__init__` | Store hyperparameters (n_estimators, base learner, random seed) |
| `_bootstrap` | Draw a bootstrap sample and return (indices, oob_indices) |
| `fit` | Train T base learners; track OOB predictions for each sample |
| `predict_proba` | Average predicted probabilities across all T models |
| `predict` | Threshold `predict_proba` at 0.5 |
| `oob_score_` | Property: accuracy on OOB samples (matches sklearn's `oob_score_`) |

### Key Design Choices
- **`np.random.choice(n, size=n, replace=True)`** — the single line that makes it bagging (not pasting)
- **OOB tracking**: maintain a `(n_train, n_estimators)` vote matrix; for each tree, only count votes for samples **not** in its bag
- **Base learner is a parameter** — bagging is agnostic to the estimator type
- **Soft voting** via averaged probabilities is more accurate than hard majority vote, as it carries confidence information


### 🧱 Scratch Implementation Code

In [ ]:
class BaggingClassifierScratch:
    """
    Bootstrap Aggregating (Bagging) Classifier — NumPy-only implementation.

    Parameters
    ----------
    base_estimator : sklearn-compatible classifier
        Any estimator with fit() / predict_proba() methods.
    n_estimators   : int, default=100
        Number of bootstrap learners to train.
    random_state   : int, default=42
        Master seed for reproducibility.

    Attributes (post-fit)
    ----------
    estimators_    : list of fitted base models
    oob_score_     : float, OOB accuracy (unbiased generalization estimate)
    """

    def __init__(self, base_estimator=None, n_estimators=100, random_state=42):
        self.base_estimator = base_estimator or DecisionTreeClassifier()
        self.n_estimators   = n_estimators
        self.random_state   = random_state
        self.estimators_    = []         # stores the T fitted models
        self._oob_pred_sum  = None       # cumulative probability per sample × class
        self._oob_pred_cnt  = None       # how many OOB votes each sample received
        self.classes_       = None
        self.oob_score_     = None

    # ── Bootstrap Helper ──────────────────────────────────────────────────────
    def _bootstrap(self, n, rng):
        """
        Draw n indices WITH replacement → bag indices.
        OOB indices = training indices NOT in the bag.

        INTERVIEW NOTE: P(sample not chosen) = (1-1/n)^n → e^{-1} ≈ 0.368
        So ~36.8% of rows are OOB for each tree — the source of the free val set.
        """
        bag_idx = rng.choice(n, size=n, replace=True)   # bootstrap sample
        oob_idx = np.setdiff1d(np.arange(n), bag_idx)   # indices NOT in bag
        return bag_idx, oob_idx

    # ── Fit ───────────────────────────────────────────────────────────────────
    def fit(self, X, y):
        """
        Train T independent base estimators on bootstrap bags.
        Simultaneously accumulate OOB predictions for a free validation estimate.
        """
        X = np.array(X)
        y = np.array(y)
        n_samples = X.shape[0]
        self.classes_ = np.unique(y)
        n_classes     = len(self.classes_)

        # OOB accumulators: shape (n_samples, n_classes)
        self._oob_pred_sum = np.zeros((n_samples, n_classes))
        self._oob_pred_cnt = np.zeros(n_samples)           # vote count per sample

        # Master RNG — ensures reproducibility while giving each tree its own seed
        rng = np.random.RandomState(self.random_state)

        for i in range(self.n_estimators):
            # 1. Bootstrap sample
            bag_idx, oob_idx = self._bootstrap(n_samples, rng)

            # 2. Clone + fit base estimator on the bag
            import copy
            estimator = copy.deepcopy(self.base_estimator)
            estimator.fit(X[bag_idx], y[bag_idx])
            self.estimators_.append(estimator)

            # 3. Accumulate OOB predictions
            # INTERVIEW NOTE: we predict ONLY on OOB samples — never on in-bag rows.
            # This is what makes OOB an unbiased estimate (no data leakage).
            if len(oob_idx) > 0:
                oob_proba = estimator.predict_proba(X[oob_idx])  # shape (|OOB|, n_classes)
                self._oob_pred_sum[oob_idx] += oob_proba
                self._oob_pred_cnt[oob_idx] += 1

        # 4. Compute OOB score
        # Only use samples that were OOB for at least one tree
        oob_mask  = self._oob_pred_cnt > 0
        oob_proba = self._oob_pred_sum[oob_mask] / self._oob_pred_cnt[oob_mask, None]
        oob_preds = self.classes_[np.argmax(oob_proba, axis=1)]
        self.oob_score_ = accuracy_score(y[oob_mask], oob_preds)

        return self

    # ── Predict Probabilities ─────────────────────────────────────────────────
    def predict_proba(self, X):
        """
        Average predicted class probabilities across all T estimators.
        Soft voting — retains confidence information; more accurate than hard vote.
        """
        X = np.array(X)
        # Stack: shape (T, n_samples, n_classes)
        all_proba = np.array([est.predict_proba(X) for est in self.estimators_])
        return all_proba.mean(axis=0)   # average over T trees → (n_samples, n_classes)

    # ── Predict ───────────────────────────────────────────────────────────────
    def predict(self, X):
        """Return hard class labels by thresholding averaged probabilities at 0.5."""
        proba = self.predict_proba(X)
        return self.classes_[np.argmax(proba, axis=1)]


print("✅ BaggingClassifierScratch class defined.")
print("   Methods: fit | predict_proba | predict | oob_score_")


### 📈 Train & Evaluate — Scratch Implementation

In [ ]:
np.random.seed(SEED)

# ── Train ─────────────────────────────────────────────────────────────────────
# Base learner: fully grown decision tree (max_depth=None) — high variance, low bias.
# This is the textbook base learner for bagging: it benefits most from variance reduction.
base_tree = DecisionTreeClassifier(max_depth=None, random_state=SEED)

scratch_model = BaggingClassifierScratch(
    base_estimator=base_tree,
    n_estimators=200,        # 200 bootstrap trees
    random_state=SEED
)

scratch_model.fit(X_train.values, y_train.values)

# ── Evaluate ──────────────────────────────────────────────────────────────────
y_pred_scratch  = scratch_model.predict(X_test.values)
y_proba_scratch = scratch_model.predict_proba(X_test.values)[:, 1]   # P(disease)

acc_scratch = accuracy_score(y_test, y_pred_scratch)
auc_scratch = roc_auc_score(y_test, y_proba_scratch)
oob_scratch = scratch_model.oob_score_

print("=" * 50)
print("   FROM-SCRATCH BAGGING — RESULTS")
print("=" * 50)
print(f"  OOB Score  : {oob_scratch:.4f}  ← free val estimate (no data wasted)")
print(f"  Test Acc   : {acc_scratch:.4f}")
print(f"  Test AUC   : {auc_scratch:.4f}")
print("=" * 50)

print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred_scratch, target_names=['No Disease', 'Disease']))

# ── Baseline Comparison ───────────────────────────────────────────────────────
single_tree = DecisionTreeClassifier(max_depth=None, random_state=SEED)
single_tree.fit(X_train, y_train)
single_acc  = accuracy_score(y_test, single_tree.predict(X_test))
single_auc  = roc_auc_score(y_test, single_tree.predict_proba(X_test)[:, 1])

print("\n📊 Bagging vs Single Tree:")
print(f"  Single Tree  — Acc: {single_acc:.4f} | AUC: {single_auc:.4f}")
print(f"  Bagging(200) — Acc: {acc_scratch:.4f} | AUC: {auc_scratch:.4f}")
delta = acc_scratch - single_acc
print(f"  Δ Accuracy   : {delta:+.4f}  {'✅ Improvement' if delta > 0 else '🔄 Comparable'}")


---
## 🛠️ Part 3 — Scikit-Learn Implementation

### How Sklearn Implements Bagging
`sklearn.ensemble.BaggingClassifier` mirrors our scratch implementation but adds:

| Feature | Detail |
|---|---|
| `n_jobs=-1` | Parallelises tree training across all CPU cores (joblib backend) |
| `oob_score=True` | Activates OOB estimation (matches our `oob_score_` property) |
| `max_samples` | Control the bag size (default 1.0 = full n; can set to e.g. 0.8 for 80%) |
| `max_features` | Feature subsampling per tree (default 1.0 = all features; set < 1 to decorrelate) |
| `bootstrap_features` | Whether to also bootstrap features (default False) |

### Key Difference from Scratch
- **Parallelism**: sklearn uses `joblib` to train trees in parallel — dramatically faster on multi-core hardware.
- **Estimator cloning**: uses `sklearn.base.clone()` internally to ensure each tree gets fresh hyperparameters.
- **OOB implementation**: sklearn accumulates votes using a similar scheme to ours but with optimised C extensions.

### Advantages of Library vs Scratch
| Criterion | Scratch | Sklearn |
|---|---|---|
| Transparency | ✅ Full visibility | ❌ Abstracted |
| Speed | ❌ Slow (Python loops) | ✅ Fast (C extensions, parallel) |
| Production readiness | ❌ | ✅ |
| Custom base learners | ✅ Any sklearn-compatible | ✅ Any sklearn-compatible |
| Learning value | ✅ High | Medium |


### 🤖 Sklearn Bagging — Train, Evaluate & Compare

In [ ]:
np.random.seed(SEED)

# ── Sklearn BaggingClassifier ─────────────────────────────────────────────────
sklearn_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=None, random_state=SEED),
    n_estimators=200,          # same T as scratch for fair comparison
    max_samples=1.0,           # full bootstrap bags
    max_features=1.0,          # use all features (vanilla bagging, not Random Forest)
    bootstrap=True,            # sampling WITH replacement
    bootstrap_features=False,  # don't bootstrap features (that's Random Forest territory)
    oob_score=True,            # free OOB validation
    n_jobs=-1,                 # parallel across all cores
    random_state=SEED
)

sklearn_model.fit(X_train, y_train)

# ── Evaluate ──────────────────────────────────────────────────────────────────
y_pred_sk  = sklearn_model.predict(X_test)
y_proba_sk = sklearn_model.predict_proba(X_test)[:, 1]

acc_sk = accuracy_score(y_test, y_pred_sk)
auc_sk = roc_auc_score(y_test, y_proba_sk)
oob_sk = sklearn_model.oob_score_

print("=" * 55)
print("   SCIKIT-LEARN BAGGING — RESULTS")
print("=" * 55)
print(f"  OOB Score  : {oob_sk:.4f}")
print(f"  Test Acc   : {acc_sk:.4f}")
print(f"  Test AUC   : {auc_sk:.4f}")
print("=" * 55)

print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred_sk, target_names=['No Disease', 'Disease']))

# ── Head-to-Head Comparison ───────────────────────────────────────────────────
print("\n📊 Implementation Comparison (n_estimators=200, same SEED):")
print(f"  {'Method':<20} {'OOB':>8} {'Acc':>8} {'AUC':>8}")
print("  " + "-" * 48)
print(f"  {'Single Tree':<20} {'N/A':>8} {single_acc:>8.4f} {single_auc:>8.4f}")
print(f"  {'Scratch Bagging':<20} {oob_scratch:>8.4f} {acc_scratch:>8.4f} {auc_scratch:>8.4f}")
print(f"  {'Sklearn Bagging':<20} {oob_sk:>8.4f} {acc_sk:>8.4f} {auc_sk:>8.4f}")
print("\n💡 Note: Minor differences between scratch and sklearn are expected due to")
print("   internal parallelism and floating-point ordering in sklearn.")


---
## 📊 Part 3 — Visualisations
Four plots:
1. **Confusion Matrices** — Scratch vs Sklearn side-by-side
2. **ROC Curves** — Single Tree vs Scratch Bagging vs Sklearn Bagging
3. **OOB Score vs n_estimators** — shows convergence and diminishing returns
4. **Feature Importance** — permutation-style from a companion Random Forest


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Plot 1: Confusion Matrices ────────────────────────────────────────────────
for ax, (title, y_pred) in zip(axes, [
    ('From Scratch', y_pred_scratch),
    ('Scikit-Learn', y_pred_sk)
]):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['No Disease', 'Disease'],
        yticklabels=['No Disease', 'Disease'],
        ax=ax, linewidths=0.5
    )
    ax.set_title(f'Confusion Matrix — {title}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

plt.suptitle('Bagging: Confusion Matrices', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot1_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("📌 Saved: plot1_confusion_matrices.png")


In [ ]:
# ── Plot 2: ROC Curves ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

for label, y_proba in [
    ('Single Decision Tree', single_tree.predict_proba(X_test)[:, 1]),
    ('Bagging — From Scratch', y_proba_scratch),
    ('Bagging — Sklearn', y_proba_sk),
]:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_val     = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, lw=2, label=f'{label} (AUC = {auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier (AUC = 0.500)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Single Tree vs Bagging', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plot2_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("📌 Saved: plot2_roc_curves.png")


In [ ]:
# ── Plot 3: OOB Score vs n_estimators ────────────────────────────────────────
# This demonstrates the "diminishing returns" property of adding more trees.
# The OOB score converges — doubling T from 500→1000 gives far less gain than 10→50.

estimator_range = [1, 5, 10, 20, 30, 50, 75, 100, 150, 200, 300]
oob_scores = []

for n in estimator_range:
    m = BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=None, random_state=SEED),
        n_estimators=n,
        oob_score=True,
        bootstrap=True,
        random_state=SEED,
        n_jobs=-1
    )
    m.fit(X_train, y_train)
    oob_scores.append(m.oob_score_)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(estimator_range, oob_scores, marker='o', color='steelblue', lw=2, markersize=6)
ax.axhline(y=max(oob_scores), color='tomato', linestyle='--', alpha=0.7,
           label=f'Max OOB = {max(oob_scores):.4f}')
ax.fill_between(estimator_range, min(oob_scores)*0.995, oob_scores,
                alpha=0.08, color='steelblue')
ax.set_xlabel('Number of Estimators (T)', fontsize=12)
ax.set_ylabel('OOB Accuracy Score', fontsize=12)
ax.set_title('OOB Score vs Number of Trees — Diminishing Returns', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Annotation
best_n = estimator_range[np.argmax(oob_scores)]
ax.annotate(f'T={best_n}', xy=(best_n, max(oob_scores)),
            xytext=(best_n + 20, max(oob_scores) - 0.01),
            arrowprops=dict(arrowstyle='->', color='gray'), fontsize=10, color='gray')

plt.tight_layout()
plt.savefig('plot3_oob_vs_trees.png', dpi=150, bbox_inches='tight')
plt.show()
print("📌 Saved: plot3_oob_vs_trees.png")
print(f"\n💡 Observation: OOB score largely plateaus after ~{best_n} trees.")
print("   This is the 'diminishing returns' property — doubling from 200→400 buys little.")


In [ ]:
# ── Plot 4: Feature Importance (from Random Forest — Bagging's sibling) ─────
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#2196F3' if v < importances.quantile(0.75) else '#F44336' for v in importances]
ax.barh(importances.index, importances.values, color=colors, edgecolor='white')
ax.set_xlabel('Mean Decrease in Impurity (MDI)', fontsize=12)
ax.set_title('Feature Importances — Random Forest (Bagging + Feature Subsampling)',
             fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Mark the top feature
top_feat = importances.idxmax()
ax.axvline(importances.max(), color='red', linestyle='--', alpha=0.4)
ax.text(importances.max() + 0.001, len(importances) - 1.5,
        f'Top: {top_feat}', color='red', fontsize=9)

plt.tight_layout()
plt.savefig('plot4_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("📌 Saved: plot4_feature_importance.png")
print(f"\n🔍 Top feature: '{top_feat}' (thalach = max heart rate achieved)")
print("   CAUTION: MDI importance can be inflated for high-cardinality features.")
print("   Use permutation importance (sklearn.inspection) for a robust alternative.")


---
## ⚗️ Part 4 — Hyperparameter Experiments

### Key Hyperparameters in Bagging

| Parameter | Default | Effect on Variance | Effect on Bias | Practical Tip |
|---|---|---|---|---|
| `n_estimators` | 10 | ↓ as T ↑ (diminishing returns) | None | Start at 100–300; profile runtime |
| `max_samples` | 1.0 | ↓ with smaller bags | ↑ slightly | Use < 1.0 on huge datasets to speed up |
| `max_features` | 1.0 | ↓ decorrelates trees | ↑ individual tree quality drops | < 1.0 approaches Random Forest |
| `max_depth` (base tree) | None | ↓ as depth ↑ (more variance to reduce) | ↑ as depth ↓ | Deep trees maximise bagging benefit |
| `bootstrap` | True | Diversity driven by replacement | — | Set False for "Pasting" (sampling w/o replacement) |

### Experiment Plan
1. **Experiment A** — `max_samples` (bag size): [0.3, 0.5, 0.7, 0.9, 1.0] → how much data per bag affects OOB stability
2. **Experiment B** — `max_features` (feature subsampling): [0.3, 0.5, 0.7, 1.0] → transition from Random Forest to vanilla Bagging


### 🔬 Hyperparameter Experiment Code

In [ ]:
np.random.seed(SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# ── Experiment A: max_samples ─────────────────────────────────────────────────
max_samples_range = [0.3, 0.5, 0.7, 0.9, 1.0]
results_A = {'max_samples': [], 'cv_acc_mean': [], 'cv_acc_std': []}

print("Experiment A: Varying max_samples (bag size fraction)")
print(f"  {'max_samples':>12} {'CV Acc':>10} {'Std':>8}")
print("  " + "-"*34)

for ms in max_samples_range:
    model = BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=None, random_state=SEED),
        n_estimators=150,
        max_samples=ms,
        max_features=1.0,
        bootstrap=True,
        random_state=SEED,
        n_jobs=-1
    )
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    results_A['max_samples'].append(ms)
    results_A['cv_acc_mean'].append(scores.mean())
    results_A['cv_acc_std'].append(scores.std())
    print(f"  {ms:>12.1f} {scores.mean():>10.4f} {scores.std():>8.4f}")

# ── Experiment B: max_features ────────────────────────────────────────────────
max_features_range = [0.3, 0.5, 0.7, 1.0]
results_B = {'max_features': [], 'cv_acc_mean': [], 'cv_acc_std': []}

print("\nExperiment B: Varying max_features (feature subsampling)")
print(f"  {'max_features':>14} {'CV Acc':>10} {'Std':>8}")
print("  " + "-"*36)

for mf in max_features_range:
    model = BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=None, random_state=SEED),
        n_estimators=150,
        max_samples=1.0,
        max_features=mf,       # feature subsampling = decorrelation
        bootstrap=True,
        bootstrap_features=False,
        random_state=SEED,
        n_jobs=-1
    )
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    results_B['max_features'].append(mf)
    results_B['cv_acc_mean'].append(scores.mean())
    results_B['cv_acc_std'].append(scores.std())
    print(f"  {mf:>14.1f} {scores.mean():>10.4f} {scores.std():>8.4f}")


In [ ]:
# ── Visualise Hyperparameter Experiments ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Panel A: max_samples ──────────────────────────────────────────────────────
ax = axes[0]
means = np.array(results_A['cv_acc_mean'])
stds  = np.array(results_A['cv_acc_std'])
xvals = results_A['max_samples']

ax.plot(xvals, means, marker='s', color='steelblue', lw=2, markersize=7, label='CV Accuracy')
ax.fill_between(xvals, means - stds, means + stds, alpha=0.15, color='steelblue', label='±1 Std Dev')
ax.set_xlabel('max_samples (bag size fraction)', fontsize=11)
ax.set_ylabel('5-Fold CV Accuracy', fontsize=11)
ax.set_title('Effect of Bag Size (max_samples)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(max(means) - 0.08, max(means) + 0.03)

best_ms = xvals[np.argmax(means)]
ax.axvline(best_ms, color='red', linestyle='--', alpha=0.5)
ax.text(best_ms + 0.01, min(means - stds) + 0.005, f'Best: {best_ms}',
        color='red', fontsize=9)

# ── Panel B: max_features ─────────────────────────────────────────────────────
ax = axes[1]
means = np.array(results_B['cv_acc_mean'])
stds  = np.array(results_B['cv_acc_std'])
xvals = results_B['max_features']

ax.plot(xvals, means, marker='D', color='darkorange', lw=2, markersize=7, label='CV Accuracy')
ax.fill_between(xvals, means - stds, means + stds, alpha=0.15, color='darkorange', label='±1 Std Dev')
ax.set_xlabel('max_features (fraction of features per tree)', fontsize=11)
ax.set_ylabel('5-Fold CV Accuracy', fontsize=11)
ax.set_title('Effect of Feature Subsampling (max_features)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(max(means) - 0.08, max(means) + 0.03)

best_mf = xvals[np.argmax(means)]
ax.axvline(best_mf, color='green', linestyle='--', alpha=0.5)
ax.text(best_mf + 0.01, min(means - stds) + 0.005, f'Best: {best_mf}',
        color='green', fontsize=9)

plt.suptitle('Bagging Hyperparameter Experiments (5-Fold CV)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot5_hyperparams.png', dpi=150, bbox_inches='tight')
plt.show()
print("📌 Saved: plot5_hyperparams.png")
print("\n💡 Key Observations:")
print(f"  A. Best max_samples = {results_A['max_samples'][np.argmax(results_A['cv_acc_mean'])]}")
print(f"     Smaller bags → more diversity but each model sees less data.")
print(f"  B. Best max_features = {results_B['max_features'][np.argmax(results_B['cv_acc_mean'])]}")
print(f"     Feature subsampling decorrelates trees → lower ρ → lower variance floor.")


---
## 🎯 Part 5 — Interview Corner

---

### Q1. Why does bagging reduce variance but not bias?

**Conceptual answer:**
- **Bias** is the *systematic* error of a model — it reflects whether the base learner's hypothesis class can represent the true function.
- **Variance** is the *sensitivity* of predictions to the specific training set used.
- Bagging trains T models on T slightly different datasets and **averages** their outputs.
- Averaging cancels out *random* fluctuations (variance) but **cannot** cancel *systematic* errors (bias).
- **Formula insight:** $\text{Var}(\bar{f}) = \rho\sigma^2 + \frac{(1-\rho)\sigma^2}{T}$. The bias term is absent — bagging is a pure variance reduction tool.

**Common candidate mistake:** Claiming bagging reduces both bias and variance. It does not — an ensemble of 500 biased trees is still biased.

---

### Q2. How does Random Forest improve on vanilla bagging?

**Answer:** RF adds **random feature subsampling** at each split (not just at each tree). This reduces $\rho$ (pairwise inter-tree correlation), which directly lowers the irreducible variance floor $\rho\sigma^2$. Tuning `max_features`:
- **Lower** → more decorrelation (↓ variance) but worse individual trees (↑ bias)
- **Higher** → closer to vanilla bagging (trees are more correlated but individually stronger)
- **Defaults:** `sqrt(p)` for classification, `p/3` for regression are well-established empirical choices.

---

### Q3. You have 10M training rows and need a Random Forest in production. What are your strategies?

| Strategy | Technique |
|---|---|
| Reduce bag size | `max_samples=0.1` — 1M rows per tree is sufficient for convergence |
| Parallelise | `n_jobs=-1` — train trees on all CPU cores |
| Limit tree depth | `max_depth=15` or `min_samples_leaf=50` to reduce memory footprint |
| Approximate splits | Switch to LightGBM (`tree_method='hist'`) for histogram-based speed |
| Distributed training | Spark MLlib / Dask-ML for cluster-scale training |

---

### Q4. Compare bagging vs boosting.

| Dimension | Bagging | Boosting |
|---|---|---|
| Primary effect | ↓ Variance | ↓ Bias |
| Training order | Parallel | Sequential |
| Noise sensitivity | Robust (noise cancels across bags) | Amplifies outliers (later trees focus on them) |
| Parallelisable? | ✅ Yes | ❌ No |
| Best on | High-variance models, noisy data | Low-variance models, clean data, tabular |

---

### Q5. Your OOB score is much higher than your test score. What do you investigate?

| Root Cause | Diagnosis |
|---|---|
| **Data leakage** | Target-derived feature or future data in training? |
| **Distribution shift** | Different time periods or populations in train vs test? |
| **Preprocessing leakage** | Was the scaler/imputer fit on the full dataset before splitting? |
| **Class imbalance** | OOB not stratified — distribution differs from test |
| **Small dataset** | OOB sets are tiny and noisy; switch to stratified k-fold |

---

### ⚠️ Common Candidate Mistakes
1. Saying bagging reduces bias — **it does not**
2. Confusing OOB score (unbiased) with training accuracy (optimistic)
3. Not knowing that $\rho$ is the fundamental bottleneck — not T
4. Claiming bagging and Random Forest are identical — RF adds feature subsampling
5. Saying bagging can fix a high-bias base learner — use boosting for that


---
## ✅ Key Takeaways

- 🎯 **Bagging is a variance reduction tool, not a bias reduction tool.** Averaging T models cancels random prediction noise; it cannot remove systematic error. Always pair with a low-bias base learner (e.g., fully grown decision trees).

- 📐 **The formula $\text{Var}(\bar{f}) = \rho\sigma^2 + \frac{(1-\rho)\sigma^2}{T}$ governs everything.** As $T \to \infty$, only $\rho\sigma^2$ remains — decorrelating models (via feature subsampling in Random Forest) matters more than adding more trees once T is large.

- 🆓 **OOB evaluation is a free, unbiased generalization estimate.** Roughly 36.8% of training samples are left out of each bootstrap bag. Predicting those samples using only their OOB trees gives an unbiased test-set approximation — no separate validation split required.

- ⚡ **Bagging is trivially parallelisable.** Each tree is trained independently on its own bootstrap bag — unlike boosting, which is sequential. On a 16-core machine, `n_jobs=-1` can reduce wall-clock time by ~10–15×.

- 🏆 **Interview edge:** Know the four failure modes — (1) bagging can't fix a biased model, (2) highly correlated bags limit variance reduction, (3) returns diminish after ~200–300 trees, (4) OOB estimates break on very small datasets. Knowing *when bagging fails* distinguishes senior candidates from juniors.

---

*Notebook complete — ready for Restart & Run All.*  
*Part of the ML Study Series: Ensemble Methods → Bagging*  
*Author: Senior ML Engineer (FAANG) + MIT Educator*
